In [2]:
%load_ext autoreload
%autoreload 2
from src.features import name_features

print(name_features("record office", "rcdbr offie"))      # typos: tri and set should stay fairly high
print(name_features("fh business", "fh fh business"))     # doubled word: set should be 1.0
print(name_features("mirasol", "team air"))               # unrelated: everything low
print(name_features("", "first food"))                    # native script: name_missing = 1

{'name_missing': 0, 'name_set': 0.75, 'name_sort': 0.75, 'name_ratio': 0.75, 'name_tri': 0.14285714285714285, 'name_tok': 0.0}
{'name_missing': 0, 'name_set': 1.0, 'name_sort': 0.88, 'name_ratio': 0.88, 'name_tri': 0.9166666666666666, 'name_tok': 1.0}
{'name_missing': 0, 'name_set': 0.4, 'name_sort': 0.4, 'name_ratio': 0.4, 'name_tri': 0.0, 'name_tok': 0.0}
{'name_missing': 1, 'name_set': 0.0, 'name_sort': 0.0, 'name_ratio': 0.0, 'name_tri': 0.0, 'name_tok': 0.0}


In [3]:
%autoreload 2
from src.normalize import norm_address
from src.features import addr_features

def t(a, b):
    print(addr_features(norm_address(a), norm_address(b)))

t("88 Olive Circle, Lebanon, TN", "88 OLIVE CIR, LEBANON, TN")                     # all high, num_match 1
t("7149, D-7 Vasant Kunj, New Delhi, Delhi", "07149, D-7 Vasant Kunj, New Delhi, DL")  # num_match 1 (zero stripped)
t("1302 Swift Creek Road, Wilson's Mills, NC", "302 Swift Creek Rd, Wilm, North Carolina")  # num_conflict 1, words high
t("342 Portland Boulevard, Portland, TN", float("nan"))                              # addr_missing 1

{'addr_missing': 0, 'addr_jaccard': 1.0, 'addr_min': 1.0, 'num_match': 1, 'num_conflict': 0, 'word_jaccard': 1.0}
{'addr_missing': 0, 'addr_jaccard': 0.6666666666666666, 'addr_min': 0.8571428571428571, 'num_match': 1, 'num_conflict': 0, 'word_jaccard': 0.8333333333333334}
{'addr_missing': 0, 'addr_jaccard': 0.45454545454545453, 'addr_min': 0.7142857142857143, 'num_match': 0, 'num_conflict': 1, 'word_jaccard': 0.5555555555555556}
{'addr_missing': 1, 'addr_jaccard': 0.0, 'addr_min': 0.0, 'num_match': 0, 'num_conflict': 0, 'word_jaccard': 0.0}


In [4]:
import sys; sys.path.append("..")
import time
import pandas as pd
from src.normalize import norm_name, norm_address
from src.blocking import build_index, get_candidates_topk, prepare_pool, rerank
from src.features import pair_features

D = "../dataset/dev/"
s1_dev = pd.read_csv(D + "s1_dev.tsv", sep="\t")
gt_dev = pd.read_csv(D + "gt_dev.tsv", sep="\t")
train_ids = set(pd.read_csv(D + "train_ids.txt", header=None)[0])
pool_df = pd.concat([pd.read_csv(D + "s2_dev.tsv", sep="\t"),
                     pd.read_csv(D + "s3_dev.tsv", sep="\t")], ignore_index=True)

# WHY build once: indexing and cleaning the pool is the slow part
index = build_index(pool_df)
prepared = prepare_pool(pool_df)

truth = {sid: (set(m.split(",")) if isinstance(m, str) else set())
         for sid, m in zip(gt_dev["source1_entity_id"], gt_dev["matched_entity_ids"])}

def build_pairs(s1_rows, k=30):
    rows = []
    for eid, name, addr, country in zip(s1_rows["entity_id"], s1_rows["business_name"],
                                        s1_rows["business_address"], s1_rows["country"]):
        first = get_candidates_topk(name, addr, country, index, 1000, 100)
        top = rerank(name, addr, first, prepared, k)
        n1, a1 = norm_name(name), norm_address(addr)  # WHY outside loop: same S1 side for all candidates
        for cid in top:
            n2, a2 = prepared[cid]
            f = pair_features(n1, a1, n2, a2)
            f["s1"], f["cand"] = eid, cid
            f["label"] = int(cid in truth[eid])  # WHY: 1 only if it is a true match
            rows.append(f)
    return pd.DataFrame(rows)

sub = s1_dev[s1_dev["entity_id"].isin(train_ids)].sample(2000, random_state=0)
t0 = time.time()
pairs = build_pairs(sub)
print("seconds:", round(time.time() - t0, 1))
print("shape:", pairs.shape, "| positive rate:", round(pairs["label"].mean(), 3))
print(pairs.groupby("label").mean(numeric_only=True).round(2).T)

seconds: 1.9
shape: (58120, 15) | positive rate: 0.105
label            0     1
name_missing  0.00  0.05
name_set      0.59  0.90
name_sort     0.51  0.85
name_ratio    0.51  0.85
name_tri      0.21  0.76
name_tok      0.19  0.72
addr_missing  0.02  0.03
addr_jaccard  0.11  0.72
addr_min      0.21  0.86
num_match     0.36  0.85
num_conflict  0.50  0.04
word_jaccard  0.11  0.73


In [5]:
val_ids = set(pd.read_csv(D + "val_ids.txt", header=None)[0])

t0 = time.time()
train_pairs = build_pairs(s1_dev[s1_dev["entity_id"].isin(train_ids)])
val_pairs = build_pairs(s1_dev[s1_dev["entity_id"].isin(val_ids)])
print("seconds:", round(time.time() - t0, 1))

# WHY save: rebuilding costs minutes, reloading costs seconds
train_pairs.to_pickle(D + "train_pairs.pkl")
val_pairs.to_pickle(D + "val_pairs.pkl")

# WHY: positives in val_pairs divided by all true matches is the recall ceiling for the matcher
val_true = sum(len(truth[e]) for e in val_ids)
print("train:", train_pairs.shape, "| positive rate:", round(train_pairs["label"].mean(), 3))
print("val:", val_pairs.shape, "| positive rate:", round(val_pairs["label"].mean(), 3))
print("recall ceiling on val:", round(val_pairs["label"].sum() / val_true, 4))

seconds: 98.0
train: (2309975, 15) | positive rate: 0.107
val: (577150, 15) | positive rate: 0.108
recall ceiling on val: 0.8968
